In [0]:
import requests
import json
import os
from datetime import datetime

In [0]:
BASE_PATH = "dbfs:/Volumes/workspace/project_data_football_raw/pontuacao_raw"

os.makedirs(BASE_PATH, exist_ok=True)

In [0]:
def run():
    print("Iniciando ingestão dinâmica de pontuação...")
    
    rodada = 1
    continuar = True
    
    while continuar:
        url = f"https://api.cartola.globo.com/atletas/pontuados/{rodada}"
        response = requests.get(url)
        
        if response.status_code == 200:
            data = response.json()
            atletas = data.get("atletas")
            
            # Se a rodada tem atletas pontuados, salvamos
            if atletas and len(atletas) > 0:
                path = f"{BASE_PATH}/rodada={rodada}"
                dbutils.fs.mkdirs(path)
                
                file_name = f"{path}/data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
                dbutils.fs.put(file_name, json.dumps(data), overwrite=True)
                
                print(f"Sucesso: Pontuação da Rodada {rodada} salva.")
                rodada += 1
            else:
                # Se vier um JSON vazio ou sem atletas, significa que chegamos na rodada futura
                print(f"Finalizado: Rodada {rodada} ainda não possui pontuações.")
                continuar = False
        else:
            print(f"Parada: Erro ou fim das pontuações na {rodada}. Status: {response.status_code}")
            continuar = False

    print("Pipeline de pontuação concluído!")
run()